<a href="https://colab.research.google.com/github/bierkittipong/DrugSynthMC2/blob/main/Molecular_Docking_AutoDock_Vina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Protein–Ligand Molecular Docking Pipeline

This notebook performs a reusable molecular docking workflow for
holo protein structures and user-defined ligands.

## Workflow

Holo protein PDB

↓

Detect co-crystallized ligand

↓

Identify experimental binding pocket

↓

Prepare receptor

↓

Calculate docking box automatically

↓

Enter ligand SMILES

↓

Generate 3D ligand

↓

Prepare ligand PDBQT

↓

Validate docking inputs

↓

Run AutoDock Vina

↓

Extract best pose

↓

Visualize docking

↓

Analyze interactions with PLIP

↓

Compare docked ligand with experimental ligand

↓

Export results

## Main software

- RDKit
- Meeko
- AutoDock Vina
- Open Babel
- PLIP
- py3Dmol
- pandas
- ProDy
- Gemmi

## Important note

Docking scores and predicted interactions are computational predictions.
They do not by themselves demonstrate experimental binding affinity
or biological activity.

## 1. Install required software and Python packages

Install the packages required for ligand preparation, receptor preparation,
docking, visualization, and protein–ligand interaction analysis.

In [ ]:
!pip install -q rdkit meeko gemmi prody py3Dmol openbabel-wheel plip

### Install AutoDock Vina

AutoDock Vina is installed separately because the Linux executable is
distributed as a precompiled binary.

In [ ]:
from pathlib import Path
import subprocess
import os

vina_path = Path("/usr/local/bin/vina")

if not vina_path.exists():
    url = (
        "https://github.com/ccsb-scripps/AutoDock-Vina/"
        "releases/download/v1.2.7/"
        "vina_1.2.7_linux_x86_64"
    )

    subprocess.run(
        ["wget", "-q", url, "-O", str(vina_path)],
        check=True
    )

    subprocess.run(
        ["chmod", "+x", str(vina_path)],
        check=True
    )

print("Vina:")
print(subprocess.run(
    [str(vina_path), "--version"],
    capture_output=True,
    text=True
).stdout)

## 2. Set up the docking project

Create a standard directory structure so that the notebook can be reused
for different proteins and ligands.

In [ ]:
from pathlib import Path

PROJECT = Path("docking_project")

DIRS = {
    "receptor": PROJECT / "receptor",
    "config": PROJECT / "config",
    "ligands": PROJECT / "ligands",
    "docking": PROJECT / "docking",
    "analysis": PROJECT / "analysis",
    "visualization": PROJECT / "visualization",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

print("Project structure created:\n")

for name, directory in DIRS.items():
    print(f"{name:15s} → {directory}")

## 3. Upload the holo protein structure

Upload a PDB structure containing the protein and its co-crystallized ligand.

The notebook will automatically inspect ATOM and HETATM records rather than
assuming a specific protein or ligand name.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()

pdb_files = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdb")
]

if not pdb_files:
    raise ValueError("Please upload a PDB file.")

source_pdb = pdb_files[0]

holo_pdb = DIRS["receptor"] / "holo_original.pdb"

shutil.copy2(
    source_pdb,
    holo_pdb
)

print("Holo structure saved to:")
print(holo_pdb)

### Inspect the holo structure

Count protein atoms, hetero atoms, chains, residues, and candidate
co-crystallized ligands.

In [ ]:
from collections import Counter

lines = holo_pdb.read_text().splitlines()

atom_lines = [
    line for line in lines
    if line.startswith(("ATOM", "HETATM"))
]

protein_atoms = [
    line for line in lines
    if line.startswith("ATOM")
]

hetero_atoms = [
    line for line in lines
    if line.startswith("HETATM")
]

print("=== HOLO STRUCTURE INSPECTION ===")
print(f"ATOM records:   {len(protein_atoms)}")
print(f"HETATM records: {len(hetero_atoms)}")

chains = Counter(
    line[21].strip()
    for line in protein_atoms
    if len(line) > 21
)

print("\nProtein chains:")
for chain, count in chains.items():
    print(f"  Chain {chain}: {count} atoms")

## 4. Detect the co-crystallized ligand automatically

Identify non-protein HETATM components in the holo structure.

Water, ions, and common crystallographic additives are excluded.
The remaining organic component(s) are treated as candidate ligands.

In [ ]:
from collections import defaultdict

# Common non-ligand residues
EXCLUDED_HET = {
    "HOH", "WAT", "DOD",
    "SO4", "PO4", "PEG",
    "EDO", "GOL",
    "NA", "CL", "K", "CA",
    "MG", "ZN", "MN",
    "FE", "CU"
}

hetero_groups = defaultdict(list)

for line in hetero_atoms:
    resname = line[17:20].strip()
    chain = line[21].strip()
    resnum = line[22:26].strip()

    if resname not in EXCLUDED_HET:
        key = (resname, chain, resnum)
        hetero_groups[key].append(line)

print("=== CANDIDATE CO-CRYSTALLIZED LIGANDS ===\n")

candidates = []

for key, records in hetero_groups.items():
    if len(records) >= 3:
        candidates.append((key, records))
        print(
            f"Ligand: {key[0]} | "
            f"Chain: {key[1]} | "
            f"Residue: {key[2]} | "
            f"Atoms: {len(records)}"
        )

if not candidates:
    raise ValueError(
        "No suitable co-crystallized ligand was automatically detected."
    )

### Select the co-crystallized ligand

If multiple candidate ligands are present, select the desired ligand
by index. The default is the first candidate.

In [ ]:
candidate_index = 0

ligand_key, ligand_records = candidates[candidate_index]

cofactor_name = ligand_key[0]
cofactor_chain = ligand_key[1]
cofactor_resnum = ligand_key[2]

print("Selected co-crystallized ligand:")
print(f"  Name:   {cofactor_name}")
print(f"  Chain:  {cofactor_chain}")
print(f"  Residue: {cofactor_resnum}")
print(f"  Atoms:  {len(ligand_records)}")

## 5. Identify the experimental binding pocket

Calculate the center and spatial extent of the co-crystallized ligand.

Protein residues within 4 Å of the experimental ligand are reported as
experimental binding-site residues.

In [ ]:
import numpy as np
import re

def atom_xyz(line):
    return np.array([
        float(line[30:38]),
        float(line[38:46]),
        float(line[46:54])
    ])

ligand_coords = np.array([
    atom_xyz(line)
    for line in ligand_records
])

center = ligand_coords.mean(axis=0)

print("=== EXPERIMENTAL LIGAND CENTER ===")
print(f"X = {center[0]:.3f} Å")
print(f"Y = {center[1]:.3f} Å")
print(f"Z = {center[2]:.3f} Å")

In [ ]:
# Calculate protein residues within 4 Å

residue_atoms = defaultdict(list)

for line in protein_atoms:
    chain = line[21].strip()
    resname = line[17:20].strip()
    resnum = line[22:26].strip()

    residue_id = (chain, resname, resnum)
    residue_atoms[residue_id].append(atom_xyz(line))

pocket_residues = []

for residue_id, coords in residue_atoms.items():

    coords = np.array(coords)

    distances = np.linalg.norm(
        coords[:, None, :] - ligand_coords[None, :, :],
        axis=2
    )

    min_distance = distances.min()

    if min_distance <= 4.0:
        pocket_residues.append(
            (*residue_id, min_distance)
        )

pocket_residues.sort(key=lambda x: x[3])

print("=== EXPERIMENTAL BINDING-SITE RESIDUES ===\n")

for chain, resname, resnum, distance in pocket_residues:
    print(
        f"{resname}{resnum}:{chain} "
        f"{distance:.2f} Å"
    )

## 6. Prepare the receptor

Remove the co-crystallized ligand and other HETATM records while retaining
the protein ATOM records.

Alternate-location atoms are resolved automatically by selecting the
highest-occupancy conformer.

In [ ]:
from collections import defaultdict

# --------------------------------------------------
# Determine highest-occupancy altloc per residue
# --------------------------------------------------

altloc_occupancy = defaultdict(lambda: defaultdict(float))

for line in protein_atoms:

    altloc = line[16].strip()

    if altloc:
        residue_key = (
            line[21].strip(),
            line[17:20].strip(),
            line[22:26].strip()
        )

        occupancy = float(line[54:60])

        altloc_occupancy[
            residue_key
        ][altloc] += occupancy


selected_altloc = {}

for residue, occupancies in altloc_occupancy.items():

    selected = max(
        occupancies,
        key=occupancies.get
    )

    selected_altloc[residue] = selected


# --------------------------------------------------
# Build cleaned receptor
# --------------------------------------------------

receptor_lines = []

for line in protein_atoms:

    residue_key = (
        line[21].strip(),
        line[17:20].strip(),
        line[22:26].strip()
    )

    altloc = line[16].strip()

    if altloc:

        if selected_altloc.get(residue_key) != altloc:
            continue

        # Remove altloc indicator
        line = line[:16] + " " + line[17:]

    receptor_lines.append(line)


receptor_fixed = (
    DIRS["receptor"] /
    "receptor_fixed.pdb"
)

receptor_fixed.write_text(
    "\n".join(receptor_lines) + "\nEND\n"
)

print("Prepared receptor:")
print(receptor_fixed)
print(f"Atoms retained: {len(receptor_lines)}")

## 7. Prepare receptor PDBQT with Meeko

Convert the cleaned protein structure into AutoDock-compatible PDBQT format.

The resulting receptor PDBQT will be used for docking.

In [ ]:
import subprocess

receptor_prefix = (
    DIRS["receptor"] /
    "receptor"
)

cmd = [
    "mk_prepare_receptor.py",
    "-i", str(receptor_fixed),
    "-o", str(receptor_prefix),
    "-p"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

receptor_pdbqt = (
    DIRS["receptor"] /
    "receptor.pdbqt"
)

assert receptor_pdbqt.exists()

print("\nReceptor PDBQT created:")
print(receptor_pdbqt)

## 8. Calculate the AutoDock Vina docking box

The docking box is calculated automatically from the experimental
co-crystallized ligand.

The box is expanded around the ligand so that the binding pocket
and ligand flexibility are adequately covered.

In [ ]:
# --------------------------------------------------
# Calculate ligand spatial extent
# --------------------------------------------------

ligand_min = ligand_coords.min(axis=0)
ligand_max = ligand_coords.max(axis=0)

ligand_extent = ligand_max - ligand_min

# Minimum box size
minimum_box = 20.0

box_size = np.maximum(
    ligand_extent + 10.0,
    minimum_box
)

print("=== VINA BOX ===")
print(
    f"center_x = {center[0]:.3f}"
)
print(
    f"center_y = {center[1]:.3f}"
)
print(
    f"center_z = {center[2]:.3f}"
)

print(
    f"size_x = {box_size[0]:.3f}"
)
print(
    f"size_y = {box_size[1]:.3f}"
)
print(
    f"size_z = {box_size[2]:.3f}"
)

### Save docking box configuration

In [ ]:
box_file = (
    DIRS["config"] /
    "vina_box.txt"
)

box_file.write_text(
    f"center_x = {center[0]:.3f}\n"
    f"center_y = {center[1]:.3f}\n"
    f"center_z = {center[2]:.3f}\n"
    f"size_x = {box_size[0]:.3f}\n"
    f"size_y = {box_size[1]:.3f}\n"
    f"size_z = {box_size[2]:.3f}\n"
)

print("Vina box saved to:")
print(box_file)

## 9. Interactive Ligand Input

Enter the ligand name and SMILES in the boxes below.

- **Ligand name:** e.g. Quercetin
- **SMILES:** paste the SMILES string
- Click **Save Ligand**

The ligand will automatically be saved as a `.smi` file in the project folder.

In [ ]:
# ==================================================
# PREPARE LIGAND DIRECTORY
# ==================================================

from pathlib import Path

# Create project directories if they do not already exist
PROJECT_DIR = Path("docking_project")

DIRS = {
    "receptor": PROJECT_DIR / "receptor",
    "config": PROJECT_DIR / "config",
    "ligands": PROJECT_DIR / "ligands",
    "docking": PROJECT_DIR / "docking",
    "analysis": PROJECT_DIR / "analysis",
    "visualization": PROJECT_DIR / "visualization",
}

# Create all directories
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

print("Ligand directory ready:")
print(DIRS["ligands"])

# ==================================================
# INTERACTIVE LIGAND INPUT
# ==================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------------------------
# Create input boxes
# --------------------------------------------------

ligand_name_box = widgets.Text(
    value="",
    placeholder="e.g. Quercetin",
    description="Ligand:",
    layout=widgets.Layout(width="700px")
)

smiles_box = widgets.Textarea(
    value="",
    placeholder="Paste SMILES here",
    description="SMILES:",
    layout=widgets.Layout(width="700px", height="100px")
)

save_button = widgets.Button(
    description="Save Ligand",
    button_style="success",
    icon="save"
)

output_box = widgets.Output()

# --------------------------------------------------
# Save function
# --------------------------------------------------

def save_ligand(b):

    with output_box:
        clear_output()

        ligand_name = ligand_name_box.value.strip()
        smiles = smiles_box.value.strip()

        # Validate
        if not ligand_name:
            print("❌ Please enter a ligand name.")
            return

        if not smiles:
            print("❌ Please enter a SMILES string.")
            return

        # Clean filename
        safe_name = "".join(
            c if c.isalnum() or c in "_-" else "_"
            for c in ligand_name
        )

        # Save path
        ligand_smi = (
            DIRS["ligands"] /
            f"{safe_name}.smi"
        )

        # Save SMILES
        ligand_smi.write_text(
            smiles + "\n"
        )

        # Store variables for later cells
        globals()["ligand_name"] = safe_name
        globals()["smiles"] = smiles
        globals()["ligand_smi"] = ligand_smi

        # Verify
        if ligand_smi.exists():

            print("✅ Ligand saved successfully!")
            print()
            print(f"Ligand name : {ligand_name}")
            print(f"SMILES      : {smiles}")
            print(f"File        : {ligand_smi}")
            print(f"Size        : {ligand_smi.stat().st_size} bytes")

        else:
            print("❌ Failed to save ligand.")

# --------------------------------------------------
# Connect button
# --------------------------------------------------

save_button.on_click(save_ligand)

# --------------------------------------------------
# Display interface
# --------------------------------------------------

display(
    ligand_name_box,
    smiles_box,
    save_button,
    output_box
)

## 10. Generate and optimize the ligand 3D structure

RDKit is used to:
1. Parse the SMILES
2. Add hydrogens
3. Generate a 3D conformer
4. Optimize the geometry
5. Save the resulting structure as SDF

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

mol = Chem.MolFromSmiles(smiles)

if mol is None:
    raise ValueError("Invalid SMILES.")

mol = Chem.AddHs(mol)

print(
    f"Atoms including hydrogens: "
    f"{mol.GetNumAtoms()}"
)

status = AllChem.EmbedMolecule(
    mol,
    randomSeed=42
)

if status != 0:
    raise RuntimeError(
        "RDKit could not generate a 3D conformer."
    )

optimization = AllChem.UFFOptimizeMolecule(
    mol
)

print(
    "UFF optimization status:",
    optimization
)

ligand_sdf = (
    DIRS["ligands"] /
    f"{ligand_name}_3D.sdf"
)

writer = Chem.SDWriter(
    str(ligand_sdf)
)

writer.write(mol)
writer.close()

print("\n3D ligand saved:")
print(ligand_sdf)

## 11. Prepare ligand PDBQT with Meeko

Convert the RDKit-generated 3D ligand into AutoDock Vina PDBQT format.

In [ ]:
ligand_pdbqt = (
    DIRS["ligands"] /
    f"{ligand_name}.pdbqt"
)

cmd = [
    "mk_prepare_ligand.py",
    "-i", str(ligand_sdf),
    "-o", str(ligand_pdbqt)
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert ligand_pdbqt.exists()

print("\nLigand PDBQT created:")
print(ligand_pdbqt)

## 12. Validate docking inputs

Check that both receptor and ligand PDBQT files exist and contain
valid AutoDock atom types, coordinates, and partial charges.

In [ ]:
def validate_pdbqt(path):

    lines = Path(path).read_text().splitlines()

    atom_lines = [
        line for line in lines
        if line.startswith(("ATOM", "HETATM"))
    ]

    if not atom_lines:
        raise ValueError(
            f"No atom records found in {path}"
        )

    coordinates = []

    for line in atom_lines:
        coordinates.append([
            float(line[30:38]),
            float(line[38:46]),
            float(line[46:54])
        ])

    coordinates = np.array(coordinates)

    if not np.isfinite(coordinates).all():
        raise ValueError(
            f"Invalid coordinates in {path}"
        )

    atom_types = [
        line[77:79].strip()
        for line in atom_lines
    ]

    charges = [
        float(line[70:76])
        for line in atom_lines
    ]

    print(f"\n{path}")
    print(f"Atoms: {len(atom_lines)}")
    print(
        f"Atom types: {sorted(set(atom_types))}"
    )
    print(
        f"Charges finite: "
        f"{np.isfinite(charges).all()}"
    )
    print("Validation: PASSED")


validate_pdbqt(receptor_pdbqt)
validate_pdbqt(ligand_pdbqt)

## 13. Interactive AutoDock Vina Parameters

Adjust the docking parameters below without editing the code.

### Main parameters

- **Exhaustiveness:** higher = more thorough search, but slower
- **Number of modes:** maximum number of poses to output
- **Energy range:** energy window for reported poses
- **Seed:** random seed for reproducibility
- **CPU:** number of CPU threads; 0 = let Vina determine the setting

Recommended starting values:

- Exhaustiveness = 16
- Number of modes = 20
- Energy range = 3 kcal/mol
- Seed = 42
- CPU = 0

In [ ]:
# ==================================================
# INTERACTIVE VINA PARAMETERS
# ==================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------------------------
# Parameter widgets
# --------------------------------------------------

exhaustiveness_box = widgets.IntSlider(
    value=16,
    min=1,
    max=64,
    step=1,
    description="Exhaustiveness:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

num_modes_box = widgets.IntSlider(
    value=20,
    min=1,
    max=50,
    step=1,
    description="Number of modes:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

energy_range_box = widgets.FloatSlider(
    value=3.0,
    min=0.5,
    max=10.0,
    step=0.5,
    description="Energy range (kcal/mol):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

seed_box = widgets.IntText(
    value=42,
    description="Seed:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

cpu_box = widgets.IntSlider(
    value=0,
    min=0,
    max=32,
    step=1,
    description="CPU (0 = automatic):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px")
)

# --------------------------------------------------
# Save button
# --------------------------------------------------

save_vina_button = widgets.Button(
    description="Apply Vina Parameters",
    button_style="success",
    icon="check"
)

output_box = widgets.Output()

# --------------------------------------------------
# Apply parameters
# --------------------------------------------------

def apply_vina_parameters(b):

    with output_box:
        clear_output()

        # Store parameters
        globals()["exhaustiveness"] = exhaustiveness_box.value
        globals()["num_modes"] = num_modes_box.value
        globals()["energy_range"] = energy_range_box.value
        globals()["seed"] = seed_box.value
        globals()["cpu"] = cpu_box.value

        print("✅ Vina parameters applied")
        print()
        print(f"Exhaustiveness : {exhaustiveness}")
        print(f"Number of modes: {num_modes}")
        print(f"Energy range   : {energy_range} kcal/mol")
        print(f"Seed           : {seed}")
        print(f"CPU            : {cpu}")

# --------------------------------------------------
# Connect button
# --------------------------------------------------

save_vina_button.on_click(
    apply_vina_parameters
)

# --------------------------------------------------
# Display interface
# --------------------------------------------------

display(exhaustiveness_box)
display(num_modes_box)
display(energy_range_box)
display(seed_box)
display(cpu_box)
display(save_vina_button)
display(output_box)

## 14. Run AutoDock Vina

Dock the user-defined ligand into the automatically determined
experimental binding pocket.

The Vina output and complete terminal log are saved for reproducibility.

In [ ]:
docked_pdbqt = (
    DIRS["docking"] /
    f"{ligand_name}_docked.pdbqt"
)

vina_log = (
    DIRS["docking"] /
    f"{ligand_name}_vina.log"
)

cmd = [
    str(vina_path),

    "--receptor",
    str(receptor_pdbqt),

    "--ligand",
    str(ligand_pdbqt),

    "--center_x",
    str(center[0]),

    "--center_y",
    str(center[1]),

    "--center_z",
    str(center[2]),

    "--size_x",
    str(box_size[0]),

    "--size_y",
    str(box_size[1]),

    "--size_z",
    str(box_size[2]),

    "--exhaustiveness",
    str(exhaustiveness),

    "--num_modes",
    str(num_modes),

    "--energy_range",
    str(energy_range),

    "--seed",
    str(seed),

    "--out",
    str(docked_pdbqt),
]

if cpu > 0:
    cmd.extend([
        "--cpu",
        str(cpu)
    ])

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

vina_log.write_text(
    result.stdout +
    "\n\nSTDERR:\n" +
    result.stderr
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "Vina docking failed. "
        f"See {vina_log}"
    )

print("\nDocking completed successfully.")
print("Output:", docked_pdbqt)
print("Log:", vina_log)

## 15. Extract docking results and best pose

Parse the Vina output table, identify the best-scoring pose,
and save it separately for downstream analysis.

In [ ]:
import re
import pandas as pd

log_text = vina_log.read_text()

pattern = re.compile(
    r"^\s*(\d+)\s+"
    r"(-?\d+\.\d+)\s+"
    r"(\d+\.\d+)\s+"
    r"(\d+\.\d+)",
    re.MULTILINE
)

matches = pattern.findall(log_text)

if not matches:
    raise ValueError(
        "Could not find Vina docking results."
    )

results_df = pd.DataFrame(
    matches,
    columns=[
        "mode",
        "affinity_kcal_mol",
        "rmsd_lb",
        "rmsd_ub"
    ]
)

results_df[
    [
        "mode",
        "affinity_kcal_mol",
        "rmsd_lb",
        "rmsd_ub"
    ]
] = results_df[
    [
        "mode",
        "affinity_kcal_mol",
        "rmsd_lb",
        "rmsd_ub"
    ]
].astype(float)

results_df["mode"] = (
    results_df["mode"].astype(int)
)

display(results_df)

### Save the best-scoring pose

In [ ]:
best_mode = int(
    results_df.iloc[0]["mode"]
)

docked_text = docked_pdbqt.read_text()

models = re.split(
    r"(?=MODEL\s+\d+)",
    docked_text
)

best_pose = None

for model in models:

    match = re.search(
        r"MODEL\s+(\d+)",
        model
    )

    if match:

        mode = int(match.group(1))

        if mode == best_mode:
            best_pose = model
            break

if best_pose is None:
    raise ValueError(
        "Best docking pose could not be extracted."
    )

best_pose_file = (
    DIRS["docking"] /
    f"{ligand_name}_best_pose.pdbqt"
)

best_pose_file.write_text(
    best_pose.strip() + "\nENDMDL\n"
)

print("Best pose:")
print(f"Mode: {best_mode}")
print(
    f"Affinity: "
    f"{results_df.iloc[0]['affinity_kcal_mol']:.3f} kcal/mol"
)

print("\nSaved:")
print(best_pose_file)

## 16. Prepare structures for visualization

Convert the receptor and best docking pose to PDB format for
3D visualization.

The PDBQT files remain the authoritative docking structures.

In [ ]:
visual_receptor = (
    DIRS["visualization"] /
    "receptor.pdb"
)

visual_ligand = (
    DIRS["visualization"] /
    f"{ligand_name}_best_pose.pdb"
)

subprocess.run(
    [
        "obabel",
        str(receptor_pdbqt),
        "-O",
        str(visual_receptor)
    ],
    check=True
)

subprocess.run(
    [
        "obabel",
        str(best_pose_file),
        "-O",
        str(visual_ligand)
    ],
    check=True
)

print("Visualization files created:")
print(visual_receptor)
print(visual_ligand)

## 17. Visualize experimental and docked ligands

Display:

- protein structure
- experimental co-crystallized ligand
- docked ligand
- experimental binding pocket residues

This provides a visual assessment of whether the docked ligand occupies
the experimentally observed binding site.

In [ ]:
import py3Dmol

holo_text = holo_pdb.read_text()
docked_visual_text = visual_ligand.read_text()

view = py3Dmol.view(
    width=1000,
    height=700
)

# Holo structure
view.addModel(
    holo_text,
    "pdb"
)

# Docked ligand
view.addModel(
    docked_visual_text,
    "pdb"
)

# Protein
view.setStyle(
    {"model": 0, "hetflag": False},
    {
        "cartoon": {
            "color": "lightgrey"
        }
    }
)

# Experimental ligand
view.setStyle(
    {"model": 0, "hetflag": True},
    {
        "stick": {
            "radius": 0.25,
            "color": "blue"
        }
    }
)

# Docked ligand
view.setStyle(
    {"model": 1},
    {
        "stick": {
            "radius": 0.30,
            "color": "red"
        }
    }
)

view.zoomTo(
    {
        "model": 0,
        "hetflag": True
    }
)

view.zoom(1.5)

view.show()

## 18. Analyze docked ligand–protein interactions with PLIP

Run PLIP on the protein–docked-ligand complex to identify:

- hydrogen bonds
- hydrophobic interactions
- π-stacking
- π-cation interactions
- salt bridges
- halogen bonds
- water bridges

In [ ]:
# Combine receptor and docked ligand

complex_pdb = (
    DIRS["analysis"] /
    f"{ligand_name}_complex.pdb"
)

receptor_lines = [
    line for line in visual_receptor.read_text().splitlines()
    if line.startswith(("ATOM", "HETATM"))
]

ligand_lines = [
    line for line in visual_ligand.read_text().splitlines()
    if line.startswith(("ATOM", "HETATM"))
]

complex_pdb.write_text(
    "\n".join(
        receptor_lines +
        ligand_lines +
        ["END"]
    ) + "\n"
)

print("Complex created:")
print(complex_pdb)

### Run PLIP

In [ ]:
plip_docked_dir = (
    DIRS["analysis"] /
    f"plip_{ligand_name}"
)

plip_docked_dir.mkdir(
    parents=True,
    exist_ok=True
)

result = subprocess.run(
    [
        "plip",
        "-f",
        str(complex_pdb),
        "-o",
        str(plip_docked_dir),
        "--xml",
        "--txt"
    ],
    capture_output=True,
    text=True
)

print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "PLIP analysis failed."
    )

print(
    f"\nPLIP analysis completed successfully:\n"
    f"{plip_docked_dir}"
)

## 19. Analyze the experimental co-crystallized ligand

Run PLIP on the original holo structure to determine the experimentally
observed protein–ligand interactions.

In [ ]:
plip_experimental_dir = (
    DIRS["analysis"] /
    "plip_experimental"
)

plip_experimental_dir.mkdir(
    parents=True,
    exist_ok=True
)

result = subprocess.run(
    [
        "plip",
        "-f",
        str(holo_pdb),
        "-o",
        str(plip_experimental_dir),
        "--xml",
        "--txt"
    ],
    capture_output=True,
    text=True
)

print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Experimental PLIP analysis failed."
    )

print(
    "Experimental ligand PLIP analysis completed."
)

## 20. Compare experimental and docked binding interactions

Compare the protein residues contacted by the experimental ligand
with those contacted by the docked ligand.

The analysis reports:

- shared residues
- experimental-only residues
- docked-ligand-only residues
- percentage of experimental contact residues reproduced
- shared interaction types

In [ ]:
import xml.etree.ElementTree as ET

def parse_plip_xml(xml_file):

    root = ET.parse(xml_file).getroot()

    interaction_types = {
        "hydrogen_bonds": "Hydrogen bond",
        "hydrophobic_interactions": "Hydrophobic",
        "pi_stacks": "Pi stacking",
        "pi_cation_interactions": "Pi-cation",
        "salt_bridges": "Salt bridge",
        "halogen_bonds": "Halogen bond",
        "water_bridges": "Water bridge",
    }

    interactions = []

    def get_text(element, *names):
        for name in names:
            child = element.find(name)
            if child is not None and child.text:
                return child.text
        return ""

    for tag, interaction_name in interaction_types.items():

        for container in root.iter(tag):

            for interaction in list(container):

                interactions.append({
                    "Interaction": interaction_name,
                    "Residue": get_text(
                        interaction,
                        "restype",
                        "resname"
                    ),
                    "Residue number": get_text(
                        interaction,
                        "resnr",
                        "resnum"
                    ),
                    "Chain": get_text(
                        interaction,
                        "reschain",
                        "chain"
                    ),
                    "Distance (Å)": get_text(
                        interaction,
                        "distance",
                        "dist"
                    )
                })

    return pd.DataFrame(interactions)

### Load PLIP results

In [ ]:
docked_xml = list(
    plip_docked_dir.glob("*.xml")
)[0]

experimental_xml = list(
    plip_experimental_dir.glob("*.xml")
)[0]

df_docked = parse_plip_xml(
    docked_xml
)

df_experimental = parse_plip_xml(
    experimental_xml
)

display(df_docked)
display(df_experimental)

### Compare contacted residues

In [ ]:
def add_residue_id(df):

    df = df.copy()

    df["Residue ID"] = (
        df["Residue"].astype(str).str.strip()
        + df["Residue number"].astype(str).str.strip()
        + ":"
        + df["Chain"].astype(str).str.strip()
    )

    return df


df_docked = add_residue_id(df_docked)
df_experimental = add_residue_id(
    df_experimental
)

experimental_residues = set(
    df_experimental["Residue ID"]
)

docked_residues = set(
    df_docked["Residue ID"]
)

shared = sorted(
    experimental_residues &
    docked_residues
)

experimental_only = sorted(
    experimental_residues -
    docked_residues
)

docked_only = sorted(
    docked_residues -
    experimental_residues
)

overlap = (
    len(shared) /
    len(experimental_residues) *
    100
    if experimental_residues
    else 0
)

print("=== BINDING-SITE COMPARISON ===\n")

print(
    "Experimental residues:",
    ", ".join(sorted(experimental_residues))
)

print(
    "\nDocked ligand residues:",
    ", ".join(sorted(docked_residues))
)

print(
    "\nShared residues:",
    ", ".join(shared)
    if shared else "None"
)

print(
    "\nExperimental-only residues:",
    ", ".join(experimental_only)
    if experimental_only else "None"
)

print(
    "\nDocked-only residues:",
    ", ".join(docked_only)
    if docked_only else "None"
)

print(
    f"\nExperimental contact-residue overlap: "
    f"{len(shared)}/"
    f"{len(experimental_residues)} "
    f"({overlap:.1f}%)"
)

## 21. Final docking summary

Summarize the docking score, experimental binding-site overlap,
and major interactions detected for the docked ligand.

In [ ]:
best_affinity = float(
    results_df.iloc[0]["affinity_kcal_mol"]
)

print("=" * 60)
print("FINAL DOCKING SUMMARY")
print("=" * 60)

print(f"\nLigand:")
print(f"  {ligand_name}")

print(f"\nBest Vina affinity:")
print(f"  {best_affinity:.3f} kcal/mol")

print(
    f"\nExperimental binding-site overlap:"
)

print(
    f"  {len(shared)}/"
    f"{len(experimental_residues)} "
    f"({overlap:.1f}%)"
)

print("\nShared binding-site residues:")

for residue in shared:
    print(f"  {residue}")

print("\nDocked-ligand interactions:")

if df_docked.empty:
    print("  No PLIP interactions detected.")
else:
    for _, row in df_docked.iterrows():

        distance = row["Distance (Å)"]

        if str(distance).strip():
            print(
                f"  {row['Interaction']} — "
                f"{row['Residue ID']} "
                f"({distance} Å)"
            )
        else:
            print(
                f"  {row['Interaction']} — "
                f"{row['Residue ID']}"
            )

print("\n" + "=" * 60)

## 22. Export docking results

Save the main docking, interaction, and comparison results as CSV files
for further analysis, plotting, reporting, or inclusion in a thesis.

In [ ]:
# Docking scores
results_df.to_csv(
    DIRS["analysis"] /
    f"{ligand_name}_vina_results.csv",
    index=False
)

# Docked interactions
df_docked.to_csv(
    DIRS["analysis"] /
    f"{ligand_name}_interactions.csv",
    index=False
)

# Experimental interactions
df_experimental.to_csv(
    DIRS["analysis"] /
    "experimental_interactions.csv",
    index=False
)

# Summary comparison
comparison_df = pd.DataFrame({
    "Experimental-only": pd.Series(experimental_only),
    "Shared": pd.Series(shared),
    "Docked-only": pd.Series(docked_only)
})

comparison_df.to_csv(
    DIRS["analysis"] /
    f"{ligand_name}_binding_site_comparison.csv",
    index=False
)

print("Results exported to:")
print(DIRS["analysis"])

In [ ]:
import subprocess
from pathlib import Path

complex_pdb = Path("docking_project/analysis/Quercetin_complex.pdb")
plip_output = Path("docking_project/analysis/plip_results")

assert complex_pdb.exists(), f"Missing: {complex_pdb}"

plip_output.mkdir(parents=True, exist_ok=True)

# Run PLIP
cmd = [
    "plip",
    "-f", str(complex_pdb),
    "-o", str(plip_output),
    "--xml",
    "--txt"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

if result.stdout:
    print("\nPLIP output:")
    print(result.stdout)

if result.stderr:
    print("\nPLIP messages:")
    print(result.stderr)

if result.returncode == 0:
    print("\nPLIP analysis completed successfully.")
    print("Results saved in:", plip_output)
else:
    print("\nPLIP encountered an error.")